In [1]:
# COLAB SETUP

%cd /content
!rm -rf /content/proto-tsrl
!git clone https://github.com/haiyan-wang/proto-tsrl.git /content/proto-tsrl
%cd /content/proto-tsrl

from google.colab import drive, runtime
drive.mount('/content/drive')

import sys
import os

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(project_root)

/content
Cloning into '/content/proto-tsrl'...
remote: Enumerating objects: 866, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 866 (delta 37), reused 26 (delta 22), pack-reused 808 (from 1)
Receiving objects: 100% (866/866), 44.14 MiB | 21.04 MiB/s, done.
Resolving deltas: 100% (512/512), done.
/content/proto-tsrl
Mounted at /content/drive
/content/proto-tsrl


In [2]:
import functools

import numpy as np

from sklearn.model_selection import StratifiedShuffleSplit

import torch
from torch.utils.data import TensorDataset, Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

from src.models.univariate_model import UnivariateModel
from src.models.ppg_deep_model_sft import UnivariateModelSFT
from src.utils.sampling_utils import *
from src.utils.training_utils import *
from src.utils.ppg_data_utils import *

In [3]:
# SETTINGS

SEED = 42

# device
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(DEVICE)

# data
INCLUDE_CLEAN_DATA = True
INCLUDE_SEMINOISY_DATA = True
INCLUDE_NOISY_DATA = False
# these are only used to initialize dataset
TRAIN_SET_SIZE = int(1e5)
SFT_SET_SIZE = 5000

# dataloaders
BATCH_SIZE = 128
VAL_RATIO = 0.05
N_WORKERS = 4

# contrastive sampling
# PPG data has length 800, rescaled to [0, 1]
COLLATE_FN_KWARGS = {
    'min_len': 100,
    'max_len': 300,
    'num_neg': 5,
    'min_overlap': 0.3,
    'min_var': 1e-4,
    'max_var': 1e-3
}

# schedule
N_EPOCHS_STAGE1 = 30
N_EPOCHS_STAGE2 = 15
N_EPOCHS_STAGE3 = 10
TOTAL_EPOCHS = N_EPOCHS_STAGE1 + N_EPOCHS_STAGE2 + N_EPOCHS_STAGE3

# logging
CHECKPOINT_EPOCHS = [*range(1, TOTAL_EPOCHS + 1)] # 1-indexed
SAVE_DIR = "/content/drive/MyDrive/Duke/Senior Year/Thesis/experiments/ppg/checkpoints_deep"

# architecture
MASK_PROB = 0.2
MID_WEIGHT = 0.5
PROTO_NEG_MARGIN = 0.1
PROTO_DIVERSITY_THRESHOLD = 0.2
LAMBDA_PROTO = 1.0
TEMPERATURE = 1.0
LAMBDA_REPR = 1e-3
GRADIENT_CLIP = 1.0

# parameter settings and optimizers
REPR_DIMS = [300]
MODELS = []
OPTIMIZER_DICTS = []
CKPT_PATHS = []
for dim in REPR_DIMS:
    model = UnivariateModel(representation_dimension = dim, mask_probability = MASK_PROB)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(DEVICE)
    MODELS.append(model)

    ckpt_path = f"{SAVE_DIR}/dim{dim}"
    CKPT_PATHS.append(ckpt_path)

    opt_dict = {
    "stage1": torch.optim.Adam(model.parameters(), lr = 1e-3, weight_decay = 1e-5),
    "stage2": torch.optim.Adam(model.parameters(), lr = 1e-3, weight_decay = 1e-5),
    "stage3": torch.optim.Adam(model.parameters(), lr = 1e-4, weight_decay = 1e-5),
    }
    OPTIMIZER_DICTS.append(opt_dict)

SCHEDULER = None

# sft settings
SFT_EPOCHS = 25
SFT_CHECKPOINT_EPOCHS = [*range(1, SFT_EPOCHS + 1)] # 1-indexed
SFT_SAVE_DIR = "/content/drive/MyDrive/Duke/Senior Year/Thesis/experiments/ppg/checkpoints_deep_sft"

SFT_BATCH_SIZE = 128

SFT_CRITERION = nn.BCEWithLogitsLoss()
SFT_MODELS = []
SFT_CKPT_PATHS = []
SFT_OPTIMIZERS = []
for dim in REPR_DIMS:
    sft_base_model_ckpt = torch.load(f'{SAVE_DIR}/dim{dim}/prototsrl_epoch55.pt', map_location = 'cpu')
    sft_base_model = UnivariateModel(representation_dimension = dim)
    sft_base_model.load_state_dict(sft_base_model_ckpt)
    sft_model = UnivariateModelSFT(untuned_model = sft_base_model, n_classes = 1) # binary classification - 1 logit sufficient
    if torch.cuda.device_count() > 1:
            sft_model = nn.DataParallel(sft_model)
    sft_model = sft_model.to(DEVICE)
    sft_model.train()

    opt = torch.optim.Adam(sft_model.parameters(), lr = 1e-4)

    SFT_MODELS.append(sft_model)
    SFT_CKPT_PATHS.append(f'{SFT_SAVE_DIR}/dim{dim}')
    SFT_OPTIMIZERS.append(opt)

cuda:0


In [5]:
# LOAD DATA

'''
# DO NOT RERUN - TRAIN AND SFT DATA ARE ALREADY SAVED FOR FASTER LOADING
X, y = load_data(
    file_path = '/content/drive/MyDrive/Duke/Senior Year/Thesis/data',
    clean = INCLUDE_CLEAN_DATA,
    seminoisy = INCLUDE_SEMINOISY_DATA,
    noisy = INCLUDE_NOISY_DATA,
    dataset = 'train',
    return_labels = True
)

indices = np.random.default_rng(SEED).permutation(X.shape[0])
X_train, y_train = X[indices][:TRAIN_SET_SIZE], y[indices][:TRAIN_SET_SIZE]
X_sft, y_sft = X[indices][TRAIN_SET_SIZE:TRAIN_SET_SIZE+SFT_SET_SIZE], y[indices][TRAIN_SET_SIZE:TRAIN_SET_SIZE+SFT_SET_SIZE]

np.savetxt('/content/drive/MyDrive/Duke/Senior Year/Thesis/data/ppg_X_train.csv', np.squeeze(X_train, axis = -1), delimiter = ',')
np.savetxt('/content/drive/MyDrive/Duke/Senior Year/Thesis/data/ppg_y_train.csv', y_train, delimiter = ',')
np.savetxt('/content/drive/MyDrive/Duke/Senior Year/Thesis/data/ppg_X_sft.csv', np.squeeze(X_sft, axis = -1), delimiter = ',')
np.savetxt('/content/drive/MyDrive/Duke/Senior Year/Thesis/data/ppg_y_sft.csv', y_sft, delimiter = ',')
'''

X_train = np.loadtxt('/content/drive/MyDrive/Duke/Senior Year/Thesis/data/ppg_X_train.csv', delimiter = ',')
y_train = np.loadtxt('/content/drive/MyDrive/Duke/Senior Year/Thesis/data/ppg_y_train.csv', delimiter = ',')
X_sft = np.loadtxt('/content/drive/MyDrive/Duke/Senior Year/Thesis/data/ppg_X_sft.csv', delimiter = ',')
y_sft = np.loadtxt('/content/drive/MyDrive/Duke/Senior Year/Thesis/data/ppg_y_sft.csv', delimiter = ',')

print(f'X_train shape: {X_train.shape}')
print(f'Train set positive samples: {np.sum(y_train)}')


dl_train, dl_val = make_dataloaders(
    X_train,
    y_train,
    batch_size = BATCH_SIZE,
    val_ratio = VAL_RATIO,
    seed = SEED,
    num_workers = N_WORKERS,
    collate_fn_kwargs = COLLATE_FN_KWARGS
)

# RESHAPE HERE WHEN collate_fn = None
X_sft = X_sft[:, :, np.newaxis]
X_sft = np.transpose(X_sft, (0, 2, 1))
print(f'X_sft shape: {X_sft.shape}')
print(f'SFT set positive samples: {np.sum(y_sft)}')

X_sft_tensor = torch.tensor(X_sft, dtype = torch.float32)
y_sft_tensor = torch.tensor(y_sft, dtype = torch.float32).unsqueeze(1)
sft_dataset = TensorDataset(X_sft_tensor, y_sft_tensor)
dl_sft = DataLoader(sft_dataset, batch_size = SFT_BATCH_SIZE, shuffle = True)

X_train shape: (100000, 800)
Train set positive samples: 36118.0
X_sft shape: (5000, 1, 800)
SFT set positive samples: 1842.0


In [ ]:
### TRAINING

for i, dim in enumerate(REPR_DIMS):
    print(f'TRAINING MODEL WITH REPRESENTATION DIMENSION {dim}')
    history = run_training(
        model = MODELS[i],
        train_loader = dl_train,
        val_loader = dl_val,
        device = DEVICE,
        optimizer_dict = OPTIMIZER_DICTS[i],
        epochs_stage1 = N_EPOCHS_STAGE1,
        epochs_stage2 = N_EPOCHS_STAGE2,
        epochs_stage3 = N_EPOCHS_STAGE3,
        scheduler_dict = SCHEDULER,
        mid_weight = MID_WEIGHT,
        proto_neg_margin = PROTO_NEG_MARGIN,
        proto_diversity_threshold = PROTO_DIVERSITY_THRESHOLD,
        lambda_proto = LAMBDA_PROTO,
        temperature = TEMPERATURE,
        lambda_repr = LAMBDA_REPR,
        grad_clip_norm = GRADIENT_CLIP,
        checkpoint_path = CKPT_PATHS[i],
        checkpoint_epochs = CHECKPOINT_EPOCHS,
        collector_fn = None,
        use_amp = True
    )

TRAINING MODEL WITH REPRESENTATION DIMENSION 300
Training for 55 epochs.

=== stage1 (30 epochs) ===
[stage1] epoch 1/30 | global 1/55
  train loss: 3.280838 | val loss: 1.567046
Saved checkpoint at epoch 1
[stage1] epoch 2/30 | global 2/55
  train loss: 1.401308 | val loss: 1.279106
Saved checkpoint at epoch 2
[stage1] epoch 3/30 | global 3/55
  train loss: 1.199720 | val loss: 1.120171
Saved checkpoint at epoch 3
[stage1] epoch 4/30 | global 4/55
  train loss: 1.104748 | val loss: 1.047975
Saved checkpoint at epoch 4
[stage1] epoch 5/30 | global 5/55
  train loss: 1.031424 | val loss: 1.006573
Saved checkpoint at epoch 5
[stage1] epoch 6/30 | global 6/55
  train loss: 0.999001 | val loss: 0.971000
Saved checkpoint at epoch 6
[stage1] epoch 7/30 | global 7/55
  train loss: 0.975960 | val loss: 0.986096
Saved checkpoint at epoch 7
[stage1] epoch 8/30 | global 8/55
  train loss: 0.962044 | val loss: 0.947409
Saved checkpoint at epoch 8
[stage1] epoch 9/30 | global 9/55
  train loss: 0.9

In [6]:
# SUPERVISED FINETUNING

def sft_train(
        model,
        dataloader,
        device,
        criterion,
        optimizer,
        n_epochs,
        checkpoint_epochs,
        checkpoint_path,
        log_interval = 1
    ):

    print(f'Finetuning for {n_epochs} epochs')

    for epoch in range(n_epochs):
        model.train()
        running_loss = 0.0

        for xb, yb in dataloader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)

        epoch_loss = running_loss / len(dataloader.dataset)

        if (epoch+1) % log_interval == 0:
            print(f"Epoch {epoch+1:02d}/{n_epochs} - Train Loss: {epoch_loss:.4f}")

        if checkpoint_epochs is not None and epoch+1 in set(checkpoint_epochs):
            ckpt_file = f'{checkpoint_path}/prototsrl_sft_epoch{epoch+1}.pt'
            m = model.module if hasattr(model, 'module') else model
            state_dict = m.state_dict()
            torch.save(state_dict, ckpt_file)

In [7]:
for i, dim in enumerate(REPR_DIMS):
    print(f'===== {dim}D =====')
    sft_train(
        model = SFT_MODELS[i],
        dataloader = dl_sft,
        device = DEVICE,
        criterion = SFT_CRITERION,
        optimizer = SFT_OPTIMIZERS[i],
        n_epochs = SFT_EPOCHS,
        checkpoint_epochs = SFT_CHECKPOINT_EPOCHS,
        checkpoint_path = f'{SFT_SAVE_DIR}/dim{dim}',
        log_interval = 1
    )

===== 300D =====
Finetuning for 25 epochs
Epoch 01/25 - Train Loss: 0.5937
Epoch 02/25 - Train Loss: 0.3923
Epoch 03/25 - Train Loss: 0.2988
Epoch 04/25 - Train Loss: 0.2638
Epoch 05/25 - Train Loss: 0.2380
Epoch 06/25 - Train Loss: 0.2069
Epoch 07/25 - Train Loss: 0.1999
Epoch 08/25 - Train Loss: 0.1831
Epoch 09/25 - Train Loss: 0.1707
Epoch 10/25 - Train Loss: 0.1538
Epoch 11/25 - Train Loss: 0.1483
Epoch 12/25 - Train Loss: 0.1436
Epoch 13/25 - Train Loss: 0.1331
Epoch 14/25 - Train Loss: 0.1329
Epoch 15/25 - Train Loss: 0.1224
Epoch 16/25 - Train Loss: 0.1155
Epoch 17/25 - Train Loss: 0.1109
Epoch 18/25 - Train Loss: 0.1106
Epoch 19/25 - Train Loss: 0.1084
Epoch 20/25 - Train Loss: 0.0948
Epoch 21/25 - Train Loss: 0.1109
Epoch 22/25 - Train Loss: 0.1135
Epoch 23/25 - Train Loss: 0.0835
Epoch 24/25 - Train Loss: 0.0827
Epoch 25/25 - Train Loss: 0.1014


In [8]:
runtime.unassign()